# 🚀 LoRA SFT Mini Demo — Colab 一鍵跑通

> **目標**:30 分鐘內在免費 Colab T4 GPU 跑完一個完整的 LoRA 微調 pipeline
>
> **配置**:Qwen2.5-0.5B-Instruct + PEFT LoRA(rank 8)+ alpaca 1K 樣本 + 1 epoch
>
> **預期時間**:訓練 ~5-10 min(T4)、~3 min(A100)
>
> **預期效果**:base model 對指令的遵循度顯著改善;adapter 約 4MB,可重複載入

## 為什麼這份 demo 存在?

本 repo 大部分 SFT 內容是 markdown,讀者很難判斷「我自己跑會怎樣」。這份 notebook 補上「一鍵跑通」的入口,跑完之後你能回答:

1. LoRA 為什麼比 full fine-tune 省 100× VRAM
2. PEFT 介面如何把 model 包裝起來(`get_peft_model`)
3. SFTTrainer 怎麼吃 chat template 格式的資料
4. adapter 儲存與重載的 workflow

## phantom-mesh 寫由(本 repo 整合主旨)

在 phantom-mesh runtime 中,LoRA adapter swap 是 multi-tenant serving 的核心:
- 每個 tenant 一個 adapter(4-50MB),共享同一個 base model
- Adapter 載入時間 < 1s,可動態切換
- 對應 [全景圖 #14 Multi-tenant LoRA Serving](../../../2024-2026_AI完整領域全景圖.md) 與 [面試題 04 Q9](../../../9.面試準備與職業發展/1.LLM面試題庫/04_系統設計題.md)

本 demo 是 multi-tenant serving 的最底層構件:**訓出一個 adapter**。

---

## 0️⃣ 環境檢查

確認你跑在 Colab、有 GPU、CUDA 可用。

In [ ]:
import subprocess, sys

# 檢查 GPU
print('=== nvidia-smi ===')
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv']).decode())

# 檢查 PyTorch + CUDA
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'Python: {sys.version.split()[0]}')

## 1️⃣ 安裝套件

用 2026-05 兼容版本(`transformers>=4.46`、`trl>=0.12`、`peft>=0.13`、`datasets`)。
Colab 已預裝 PyTorch,不要 reinstall 否則會 break。

In [ ]:
%%capture
!pip install -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" "datasets>=3.0" "accelerate>=1.0" "bitsandbytes>=0.43"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

print('✅ All imports OK')

## 2️⃣ 載入 base model

用 **Qwen2.5-0.5B-Instruct**——494M params、Apache-2.0、繁中能力 OK、FP16 約 1GB,T4 GPU 輕鬆載入。

想換更大 model 的可改成 `Qwen2.5-1.5B-Instruct` 或 `Qwen2.5-3B-Instruct`(後者需要 4-bit 量化才能在 T4 跑;見最後一節「擴展」)。

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)

print(f'Base model: {MODEL_NAME}')
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'VRAM 占用: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 3️⃣ 微調前 baseline:看 base model 怎麼回答

用同樣的 prompt 在訓練前後比較,最直觀感受 LoRA 效果。

In [ ]:
def chat(prompt: str, max_new_tokens: int = 120) -> str:
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

TEST_PROMPTS = [
    'Give three tips for staying healthy.',
    'Write a short poem about machine learning.',
]

print('=== Baseline(訓練前)===')
for p in TEST_PROMPTS:
    print(f'\n▶ Q: {p}')
    print(f'  A: {chat(p)}')

## 4️⃣ 載入並格式化資料

用 [`tatsu-lab/alpaca`](https://huggingface.co/datasets/tatsu-lab/alpaca) 前 1000 筆。
格式統一為 ChatML(Qwen 用的 chat template 自動處理)。

想換中文:`shareAI/ShareGPT-Chinese-English-90k`、`silk-road/alpaca-data-gpt4-chinese`、`m-a-p/COIG-CQIA` 都可。

In [ ]:
raw = load_dataset('tatsu-lab/alpaca', split='train[:1000]')

def to_chat(example):
    # alpaca 格式:instruction + input(optional)+ output
    user = example['instruction']
    if example['input'].strip():
        user += '\n\n' + example['input']
    messages = [
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': example['output']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = raw.map(to_chat, remove_columns=raw.column_names)
print(f'資料筆數: {len(dataset)}')
print(f'\n第一筆 sample:\n{dataset[0]["text"][:600]}...')

## 5️⃣ 設定 LoRA

**關鍵超參數**:
- `r=8`:rank,愈大愈像 full fine-tune(也愈貴)。8 適合 0.5-3B model;對 7B+ 模型常設 16-64
- `lora_alpha=16`:scaling factor,通常設 `2×r`
- `target_modules`:Qwen 系列預設打在 `q_proj, k_proj, v_proj, o_proj`(attention 4 條),加 MLP 的 `gate_proj, up_proj, down_proj` 效果更好但成本翻倍
- `lora_dropout=0.05`:防 overfit

對比 full FT:base 494M → LoRA 可訓參數約 1.7M(**0.34%**),VRAM 從 ~4GB 降到 ~1.5GB。

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()
# 預期: trainable params: 1,720,320 || all params: 495,749,632 || trainable%: 0.347%

## 6️⃣ 設定 SFTTrainer 並開始訓練

T4(16GB)上跑 1000 樣本 × 1 epoch 大約 **5-10 分鐘**。
想跑更快可改 `num_train_epochs=1` + `max_steps=100` 提前停。

In [ ]:
training_args = SFTConfig(
    output_dir='./lora-qwen-alpaca-demo',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,  # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy='epoch',
    fp16=True,
    optim='adamw_torch',
    max_seq_length=512,
    dataset_text_field='text',
    packing=False,
    report_to='none',  # 想接 wandb 改成 'wandb'
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

## 7️⃣ 微調後評估:同樣的 prompt,再跑一次

預期看到回答:
- 更貼合「指令-回答」格式(alpaca 訓練的方向)
- 條列更清楚
- 對中性 instruction 的遵循度提升

In [ ]:
print('=== 微調後(LoRA adapter 啟用)===')
for p in TEST_PROMPTS:
    print(f'\n▶ Q: {p}')
    # peft_model 自動把 LoRA 累加到 forward
    messages = [{'role': 'user', 'content': p}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(peft_model.device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=120, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    print(f'  A: {answer}')

## 8️⃣ 儲存 adapter,重新載入

Adapter 只有 ~4MB(對比 base model 1GB),適合多 tenant 個別儲存。

In [ ]:
import os
ADAPTER_PATH = './lora-qwen-alpaca-demo/adapter'
peft_model.save_pretrained(ADAPTER_PATH)
size_mb = sum(os.path.getsize(os.path.join(ADAPTER_PATH, f)) for f in os.listdir(ADAPTER_PATH)) / 1e6
print(f'✅ Adapter 已儲存到 {ADAPTER_PATH},總大小 {size_mb:.2f} MB')
print(f'\n檔案清單:')
for f in sorted(os.listdir(ADAPTER_PATH)):
    print(f'  - {f}')

In [ ]:
# 模擬 production:從 disk 重新載入 base + adapter
del peft_model, trainer
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
reloaded = PeftModel.from_pretrained(base, ADAPTER_PATH)
reloaded.eval()

messages = [{'role': 'user', 'content': TEST_PROMPTS[0]}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(reloaded.device)
with torch.no_grad():
    out = reloaded.generate(**inputs, max_new_tokens=120, do_sample=False, pad_token_id=tokenizer.pad_token_id)
print('=== Reload + 推理測試 ===')
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

## 9️⃣ phantom-mesh 真實工程考量

Demo 完了。把上面這條路徑放到 phantom-mesh 這類 multi-tenant runtime 會撞到的真實問題:

### 9.1 Adapter swap 延遲
- LoRA adapter ~4MB,從 disk 載入 < 1s;從 S3 載入 ~2-3s
- 想 < 200ms swap:預載到 GPU RAM(`max_loras` 設定);**S-LoRA / Punica** 是這條路的 SOTA
- 對應 [全景圖 #14](../../../2024-2026_AI完整領域全景圖.md) + [面試題 04 Q9](../../../9.面試準備與職業發展/1.LLM面試題庫/04_系統設計題.md)

### 9.2 Cost tracking
- 每 user 一個 adapter,計費要 per-adapter attribution
- 訓練 cost = GPU-hour;serving cost = base model amortize + adapter 量化開銷
- phantom-mesh cost tracker 模組對應到 [Case_02 LLM Gateway §5.5](../../../9.面試準備與職業發展/2.系統設計案例/Case_02_LLM_Gateway_API_Platform.md)

### 9.3 Checkpoint resume
- 訓練到一半中斷,從 last checkpoint 重啟
- `SFTTrainer(resume_from_checkpoint=True)` 自動處理
- 對 long training 是必須

### 9.4 多 adapter 並行 serve
- vLLM 的 `--enable-lora --max-loras N` 直接支援
- 每個 request header 帶 `lora-adapter-name`
- 對應 [`../../8.模型部署與運維/vLLM_部署實戰.md`](../../8.模型部署與運維/vLLM_部署實戰.md)

### 9.5 安全
- Adapter 來自 user 上傳 → 可能含惡意 weight
- 應做 weight sanity check(NaN / 過大 norm)
- 隔離 namespace,避免 cross-tenant 汙染

---

## 🔬 擴展練習

1. **改用 QLoRA(4-bit base)**:把 `from_pretrained` 加上 `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')`,可跑 Qwen2.5-3B 在 T4 上
2. **換 DoRA**:`LoraConfig(use_dora=True, ...)` — 詳見 [`../Modern_LoRA_Variants.md`](../Modern_LoRA_Variants.md)
3. **接 DPO 偏好對齊**:訓完 SFT 後接 `DPOTrainer`,見 [`../../6.偏好對齊 (Alignment) 技術/DPO_SimPO_ORPO_對比實驗.md`](../../6.偏好對齊%20(Alignment)%20技術/DPO_SimPO_ORPO_對比實驗.md)
4. **接 GRPO 推理對齊**:見 [`../../6.偏好對齊 (Alignment) 技術/GRPO_DAPO_RLVR_實戰.md`](../../6.偏好對齊%20(Alignment)%20技術/GRPO_DAPO_RLVR_實戰.md)
5. **部署到 vLLM**:用 `vllm serve Qwen/Qwen2.5-0.5B-Instruct --enable-lora --lora-modules my=./lora-qwen-alpaca-demo/adapter` 啟動 adapter-aware serving
6. **改用中文 dataset**:`load_dataset('m-a-p/COIG-CQIA', 'ruozhiba', split='train[:1000]')`,看是否能訓出能講繁中的 model

---

## 📚 References

- [PEFT 官方文檔](https://huggingface.co/docs/peft)
- [TRL SFTTrainer 文檔](https://huggingface.co/docs/trl/sft_trainer)
- [LoRA 原論文 (Hu et al. 2021)](https://arxiv.org/abs/2106.09685)
- [QLoRA (Dettmers et al. 2023)](https://arxiv.org/abs/2305.14314)
- 本 repo:[`../進階微調策略_LoRA_QLoRA.md`](../進階微調策略_LoRA_QLoRA.md)、[`../Modern_LoRA_Variants.md`](../Modern_LoRA_Variants.md)、[`../Quantization_Primer.md`](../Quantization_Primer.md)

---

**Last updated**: 2026-05-16  
**Tested on**: Colab T4 (Free tier),Python 3.10,transformers 4.46,trl 0.12,peft 0.13